# 02 · Incremental Generator

**Purpose:** Append fresh synthetic rows to all 4 Bronze tables to simulate ongoing data growth.

**Run:** Scheduled — every 6 hours via Databricks Jobs (also works manually anytime)

**Compute:** Databricks serverless notebook.

**How it works:**
1. Reads existing user_ids from bronze_users (ensures FK integrity)
2. Generates random-but-realistic row counts per run
3. Appends to all 4 Bronze Delta tables (never overwrites)
4. Every row stamped with pipeline_run_id for lineage
5. Returns RUN_ID via dbutils.notebook.exit() for job logging

**Data growth per run (approximate):**
- New users: 5 – 20
- New transactions: 80 – 200
- New sessions: 50 – 120
- New tickets: 15 – 40

**⚠️ Do not run this before Notebook 01 has completed.**

## Step 1: Install Dependencies

Install the `faker` library to generate realistic synthetic data. The notebook kernel will restart after installation to load the new package.

In [0]:
%pip install faker --quiet
dbutils.library.restartPython()

## Step 2: Imports and Run Configuration

Import required libraries and configure the incremental run:
- **Unseeded randomness:** Each run produces different volumes to simulate real traffic fluctuation
- **Variable row counts:** Randomly chosen within ranges to mimic live data growth
- **Run ID:** Timestamp-based identifier for tracking this specific incremental load
- **Date range:** New data is timestamped within the last 48 hours to feel recent

In [0]:
import uuid
import random
from datetime import datetime, timedelta

from faker import Faker
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

fake  = Faker("en_IN")
spark = SparkSession.builder.getOrCreate()

# Intentionally unseeded — every run produces different volumes
# and different data. This is what makes it feel like live traffic.
random.seed()

RUN_TS = datetime.utcnow()
RUN_ID = RUN_TS.strftime("%Y%m%d_%H%M%S")

# Row counts vary per run — simulates real traffic fluctuation
# Adjust the ranges up/down to control how fast your dataset grows
CFG = {
    "new_users":        random.randint(5,   20),
    "new_transactions": random.randint(80,  200),
    "new_sessions":     random.randint(50,  120),
    "new_tickets":      random.randint(15,  40),
}

print(f"Run ID : {RUN_ID}")
print(f"UTC    : {RUN_TS.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Config : {CFG}")

## Step 3: Define Helper Functions and Lookup Lists

Create reusable helper functions and lookup lists for generating realistic incremental data:
- **Lookup lists:** Same categories, cities, and options as the initial load for consistency
- **rand_recent():** Generates timestamps within the last N hours to make new rows feel recent
- **Helper functions:** Date formatting, unique ID generation

In [0]:
CITIES     = ["Mumbai","Delhi","Bangalore","Chennai","Hyderabad",
              "Pune","Kolkata","Ahmedabad","Jaipur","Coimbatore"]
SEGMENTS   = ["Premium","Standard","Basic","Trial"]
CATS       = ["Electronics","Fashion","Groceries","Travel",
              "Entertainment","Health","Sports"]
PAYMENTS   = ["Credit Card","Debit Card","UPI","Net Banking","Wallet"]
DEVICES    = ["Mobile","Desktop","Tablet"]
PLATFORMS  = ["iOS","Android","Web"]
ISSUES     = ["Payment Failure","Delivery Issue","Login Problem",
              "Refund Request","Product Defect","Account Query"]
PRIORITIES = ["Low","Medium","High","Critical"]

# Timestamps within the last N hours — makes new rows feel recent
def rand_recent(hours_back: int = 48) -> datetime:
    offset = timedelta(seconds=random.randint(0, hours_back * 3600))
    return RUN_TS - offset

def fmt(dt: datetime) -> str:
    return dt.strftime("%Y-%m-%d %H:%M:%S")

def short_id(prefix: str = "") -> str:
    return prefix + str(uuid.uuid4())[:8].upper()

print("Helper functions and lookup lists defined successfully")

## Step 4: Read Existing User IDs

Pull all current user_ids from Bronze before generating anything new:
- **Why:** This guarantees every FK in new transactions, sessions, and tickets points to a real user
- **Benefit:** No orphan rows, no referential integrity issues
- **Safety check:** Raises error if bronze_users is empty (must run 01_bronze_initial_load first)

In [0]:
existing_ids = [
    r.user_id
    for r in spark.table("customer360.bronze_users")
                  .select("user_id")
                  .collect()
]

print(f"Existing users in Bronze : {len(existing_ids)}")

if len(existing_ids) == 0:
    raise RuntimeError(
        "bronze_users is empty. "
        "Run 01_bronze_initial_load first."
    )

## Step 5: Generate and Append New Users

Create new user records with recent signup dates:
- **Volume:** Random count (5-20 users per run) to simulate organic growth
- **Signup timing:** Very recent (within last 2 hours) to feel like fresh registrations
- **Active status:** All new users are active by default
- **Append mode:** Never overwrites existing data

In [0]:
new_ids  = []
new_user_rows = []

for _ in range(CFG["new_users"]):
    uid    = short_id()
    signup = rand_recent(hours_back=2)   # signed up very recently
    new_ids.append(uid)
    new_user_rows.append((
        uid,
        fake.name(),
        random.randint(18, 65),
        random.choice(["Male", "Female", "Other"]),
        signup.strftime("%Y-%m-%d"),
        random.choice(CITIES),
        "India",
        fake.email(),
        random.choice(SEGMENTS),
        1,          # new users are always active
        RUN_ID,
    ))

users_schema = StructType([
    StructField("user_id",         StringType(),  False),
    StructField("name",            StringType(),  True),
    StructField("age",             IntegerType(), True),
    StructField("gender",          StringType(),  True),
    StructField("signup_date",     StringType(),  True),
    StructField("city",            StringType(),  True),
    StructField("country",         StringType(),  True),
    StructField("email",           StringType(),  True),
    StructField("segment",         StringType(),  True),
    StructField("is_active",       IntegerType(), True),
    StructField("pipeline_run_id", StringType(),  True),
])

(
    spark.createDataFrame(new_user_rows, schema=users_schema)
    .write.format("delta")
    .mode("append")
    .saveAsTable("customer360.bronze_users")
)

print(f"Appended {len(new_user_rows)} new users")

## Step 6: Generate and Append New Transactions

Create new transaction records using existing AND newly created users:
- **User pool:** Combines existing_ids + new_ids for FK integrity
- **Amount distribution:** Intentionally skewed — most txns small (₹50-500), some medium, few large
- **Realistic patterns:** 85% success, 10% failed, 5% refunded
- **Recent timestamps:** All transactions within last 48 hours

This mirrors real e-commerce patterns where most purchases are small.

In [0]:
# Full pool = existing users + users just created this run
all_ids = existing_ids + new_ids

txn_rows = []

for _ in range(CFG["new_transactions"]):
    ts = rand_recent(hours_back=48)

    # Skewed amount distribution — realistic e-commerce pattern
    amount = round(random.choices(
        population=[
            random.uniform(50,     500),
            random.uniform(500,   5_000),
            random.uniform(5_000, 50_000),
        ],
        weights=[60, 30, 10],
    )[0], 2)

    txn_rows.append((
        short_id("TXN"),
        random.choice(all_ids),
        amount,
        fmt(ts),
        short_id("PRD"),
        random.choice(CATS),
        random.choice(PAYMENTS),
        random.choice(CITIES),
        "India",
        random.choices(
            ["success", "failed", "refunded"],
            weights=[85, 10, 5]
        )[0],
        random.choice([0, 0, 0, 5, 10, 15, 20]),
        random.choice(PLATFORMS),
        RUN_ID,
    ))

txn_schema = StructType([
    StructField("transaction_id",        StringType(),  False),
    StructField("user_id",               StringType(),  False),
    StructField("amount",                DoubleType(),  True),
    StructField("transaction_timestamp", StringType(),  True),
    StructField("product_id",            StringType(),  True),
    StructField("category",              StringType(),  True),
    StructField("payment_method",        StringType(),  True),
    StructField("city",                  StringType(),  True),
    StructField("country",               StringType(),  True),
    StructField("status",                StringType(),  True),
    StructField("discount_pct",          IntegerType(), True),
    StructField("platform",              StringType(),  True),
    StructField("pipeline_run_id",       StringType(),  True),
])

(
    spark.createDataFrame(txn_rows, schema=txn_schema)
    .write.format("delta")
    .mode("append")
    .saveAsTable("customer360.bronze_transactions")
)

print(f"Appended {len(txn_rows)} transactions")

## Step 7: Generate and Append New App Sessions

Create new session records tracking user engagement:
- **Session duration:** Random 1-90 minutes per session
- **Engagement metrics:** Pages visited, actions taken
- **Bounce detection:** Sessions under 1 minute marked as bounces
- **Device diversity:** Mix of Mobile/Desktop/Tablet across iOS/Android/Web

In [0]:
session_rows = []

for _ in range(CFG["new_sessions"]):
    start = rand_recent(hours_back=48)
    dur   = timedelta(minutes=random.randint(1, 90))
    end   = start + dur

    session_rows.append((
        short_id("SES"),
        random.choice(all_ids),
        fmt(start),
        fmt(end),
        round(dur.total_seconds() / 60, 1),
        random.randint(1, 25),
        random.randint(0, 15),
        random.choice(DEVICES),
        random.choice(PLATFORMS),
        1 if dur.total_seconds() < 60 else 0,
        RUN_ID,
    ))

session_schema = StructType([
    StructField("session_id",            StringType(),  False),
    StructField("user_id",               StringType(),  False),
    StructField("session_start",         StringType(),  True),
    StructField("session_end",           StringType(),  True),
    StructField("session_duration_mins", DoubleType(),  True),
    StructField("pages_visited",         IntegerType(), True),
    StructField("actions_taken",         IntegerType(), True),
    StructField("device_type",           StringType(),  True),
    StructField("platform",              StringType(),  True),
    StructField("is_bounce",             IntegerType(), True),
    StructField("pipeline_run_id",       StringType(),  True),
])

(
    spark.createDataFrame(session_rows, schema=session_schema)
    .write.format("delta")
    .mode("append")
    .saveAsTable("customer360.bronze_app_usage")
)

print(f"Appended {len(session_rows)} sessions")

## Step 8: Generate and Append New Support Tickets

Create new support ticket records with realistic complaint patterns:
- **Heavy complainers:** 30 users sampled to appear 3× more often (churn signal!)
- **Resolution rate:** 75% of tickets get resolved within 1-72 hours
- **Priority mix:** Low/Medium/High/Critical distributed naturally
- **Satisfaction scores:** Only assigned to resolved tickets (1-5 rating)

In [0]:
# Bias pool — heavy complainers appear 3x more often
heavy_complainers = random.sample(existing_ids, min(30, len(existing_ids)))
ticket_pool       = heavy_complainers * 3 + all_ids

ticket_rows = []

for _ in range(CFG["new_tickets"]):
    created  = rand_recent(hours_back=48)
    resolved = (
        created + timedelta(hours=random.randint(1, 72))
        if random.random() > 0.25     # 75% of tickets get resolved
        else None
    )

    ticket_rows.append((
        short_id("TKT"),
        random.choice(ticket_pool),
        random.choice(ISSUES),
        random.choice(PRIORITIES),
        fmt(created),
        fmt(resolved) if resolved else None,
        round((resolved - created).total_seconds() / 3600, 1) if resolved else None,
        "closed" if resolved else "open",
        random.randint(1, 5) if resolved else None,
        RUN_ID,
    ))

ticket_schema = StructType([
    StructField("ticket_id",          StringType(),  False),
    StructField("user_id",            StringType(),  False),
    StructField("issue_type",         StringType(),  True),
    StructField("priority",           StringType(),  True),
    StructField("created_at",         StringType(),  True),
    StructField("resolved_at",        StringType(),  True),
    StructField("resolution_hours",   DoubleType(),  True),
    StructField("status",             StringType(),  True),
    StructField("satisfaction_score", IntegerType(), True),
    StructField("pipeline_run_id",    StringType(),  True),
])

(
    spark.createDataFrame(ticket_rows, schema=ticket_schema)
    .write.format("delta")
    .mode("append")
    .saveAsTable("customer360.bronze_support_tickets")
)

print(f"Appended {len(ticket_rows)} tickets")

## Step 9: Verify Row Counts Grew

Confirm that rows were successfully appended to all tables:
- Group by pipeline_run_id to see exactly which rows came from which run
- Verify the current run's RUN_ID appears in the results
- Check that row counts match expected volumes

In [0]:
%sql
-- Confirm append worked — group by pipeline_run_id to see
-- exactly which rows came from which run
SELECT
    'bronze_users'           AS table_name,
    pipeline_run_id,
    COUNT(*)                 AS rows_in_run
FROM customer360.bronze_users
GROUP BY pipeline_run_id
ORDER BY pipeline_run_id DESC
LIMIT 5;

In [0]:
%sql
SELECT
    'bronze_transactions'    AS table_name,
    pipeline_run_id,
    COUNT(*)                 AS rows_in_run
FROM customer360.bronze_transactions
GROUP BY pipeline_run_id
ORDER BY pipeline_run_id DESC
LIMIT 5;

## Step 10: Run Summary and Exit

Generate final run summary showing:
- **Added this run:** Counts for each table
- **Total rows:** Current totals across all tables
- **RUN_ID:** Returned via dbutils.notebook.exit() for job logging

The RUN_ID is passed back to the Databricks Job for tracking and downstream pipeline orchestration.

In [0]:
totals = {
    t: spark.table(f"customer360.{t}").count()
    for t in [
        "bronze_users",
        "bronze_transactions",
        "bronze_app_usage",
        "bronze_support_tickets",
    ]
}

print("=" * 58)
print(f"  Run ID  : {RUN_ID}")
print(f"  Added   : +{len(new_user_rows)} users  "
      f"+{len(txn_rows)} txns  "
      f"+{len(session_rows)} sessions  "
      f"+{len(ticket_rows)} tickets")
print("-" * 58)
for table, count in totals.items():
    print(f"  {table:<35} {count:>7} total rows")
print("=" * 58)
print("  Next in pipeline : 03_silver_transform")

# Return RUN_ID to the Databricks Job — visible in job run logs
dbutils.notebook.exit(RUN_ID)